# Employee Data Engineering and EDA

This notebook collects employee data from the cloud database, checks its quality, prepares analysis-friendly columns, engineers features, and scales numeric values.

## Data Collection

Employee records are sourced from the cloud PostgreSQL database configured by the `DATABASE_URL` environment variable. The query selects the complete `employees` table using `psycopg2`, and the returned rows and column names are loaded into a Pandas DataFrame.

In [1]:
import os
from pathlib import Path
import pandas as pd
import psycopg2

In [2]:
from dotenv import load_dotenv
BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / '.env')
DATABASE_URL = os.getenv('DATABASE_URL')

with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute('SELECT * FROM employees')
        rows = cursor.fetchall()
        columns = [column[0] for column in cursor.description]

df = pd.DataFrame(rows, columns=columns)
df.head()

,employee_id,name,start_date,salary,department_id
0,60,Elizabeth Peterson,2019-01-15,169868,3
1,68,Amanda Weaver,2019-12-11,135953,3
2,76,Tina Tanner,2019-12-07,109270,9
3,77,Brian Perry,2015-12-29,159882,1
4,80,Craig Maldonado,2021-07-26,61523,1


## Data Dirty Update

This transaction selects four random employee rows, records their IDs, and introduces invalid-looking values in `name` and `position`, plus suspicious boundary values in `start_date` and `salary`. The view cell uses these recorded IDs so it displays only rows changed by the current run. Neon enforces non-null values, valid dates from 2015-01-01 through 2024-12-31, and salaries from 60,000 through 200,000.

In [3]:
with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT employee_id FROM employees ORDER BY random() LIMIT 4"
        )
        dirty_employee_ids = [row[0] for row in cursor.fetchall()]

        cursor.execute(
            "UPDATE employees SET name = '12345' WHERE employee_id = %s",
            (dirty_employee_ids[0],),
        )
        cursor.execute(
            "UPDATE employees SET position = '???' WHERE employee_id = %s",
            (dirty_employee_ids[1],),
        )
        cursor.execute(
            "UPDATE employees SET start_date = '2015-01-01' WHERE employee_id = %s",
            (dirty_employee_ids[2],),
        )
        cursor.execute(
            "UPDATE employees SET salary = 60000 WHERE employee_id = %s",
            (dirty_employee_ids[3],),
        )
        cursor.execute(
            """
            INSERT INTO employees (employee_id, name, position, start_date, salary)
            SELECT 101, name, position, start_date, salary
            FROM employees
            WHERE employee_id = 6
            ON CONFLICT (employee_id) DO NOTHING
            """
        )
        dirty_employee_ids.append(101)

print('Dirty employee IDs:', dirty_employee_ids)
print('Random dirty test data added to the Neon employees table.')

UndefinedColumn: column "position" of relation "employees" does not exist
LINE 1: UPDATE employees SET position = '???' WHERE employee_id = 64
                             ^


## View Data Dirty

In [ ]:
with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT
                employee_id,
                name,
                position,
                start_date,
                salary
            FROM employees
            WHERE employee_id = ANY(%s)
            ORDER BY employee_id
            """,
            (dirty_employee_ids,),
        )
        dirty_rows = cursor.fetchall()
        dirty_columns = [column[0] for column in cursor.description]

dirty_df = pd.DataFrame(dirty_rows, columns=dirty_columns)
dirty_df

,employee_id,name,position,start_date,salary
0,9,David Donaldson,Marketing Manager,2015-01-01,197539
1,23,12345,???,2017-05-11,165589
2,25,Douglas Blake,???,2023-01-29,170136
3,53,Timothy Mccann,Accountant,2024-12-13,60000
4,101,Amy Jordan,Logistics Coordinator,2024-05-10,71529


## Descriptive Statistics

The following checks summarize the current employee dataset before cleaning. They show the dataset shape, column types and non-null counts, numeric distributions, and missing-value counts.

In [ ]:
with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute('SELECT * FROM employees')
        stats_rows = cursor.fetchall()
        stats_columns = [column[0] for column in cursor.description]

stats_df = pd.DataFrame(stats_rows, columns=stats_columns)

stats_df.info()
display(stats_df.describe(include='all').T)
display(stats_df.isnull().sum().to_frame(name='missing_count'))

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   employee_id  101 non-null    int64 
 1   name         101 non-null    str   
 2   position     101 non-null    str   
 3   start_date   101 non-null    object
 4   salary       101 non-null    int64 
dtypes: int64(2), object(1), str(2)
memory usage: 4.1+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
employee_id,101.0,NaN,NaN,NaN,51.0,29.300171,1.0,26.0,51.0,76.0,101.0
name,101,98,12345,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
position,101,12,Accountant,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
start_date,101,97,2015-01-01,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
salary,101.0,NaN,NaN,NaN,121273.227723,40538.661981,60000.0,88698.0,115438.0,154398.0,200000.0


,missing_count
employee_id,0
name,0
position,0
start_date,0
salary,0


## Data Cleaning

The cleaning step reloads the current Neon data, normalizes blank text, validates names and job titles, converts dates, checks salary bounds, and removes duplicate-like records. Boundary dates and salaries are retained because they satisfy Neon’s database constraints.

In [ ]:
with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute('SELECT * FROM employees')
        rows = cursor.fetchall()
        columns = [column[0] for column in cursor.description]

df = pd.DataFrame(rows, columns=columns)
df = df.replace(r'^\s*$', pd.NA, regex=True)
df['name'] = df['name'].astype('string').str.strip()
df['position'] = df['position'].astype('string').str.strip().str.title()
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')

valid_name = df['name'].str.fullmatch(r'[A-Za-z]+(?:[ -][A-Za-z]+)*', na=False)
valid_positions = {
    'Accountant',
    'Hr Specialist',
    'Marketing Manager',
    'Sales Representative',
    'Financial Analyst',
    'Operations Manager',
    'Graphic Designer',
    'Customer Support Representative',
    'Logistics Coordinator',
    'Executive Assistant',
}
valid_position = df['position'].isin(valid_positions)
valid_salary = df['salary'].between(60000, 200000)

print('Invalid names:', (~valid_name).sum())
print('Invalid positions:', (~valid_position).sum())
print('Invalid dates:', df['start_date'].isna().sum())
print('Invalid salaries:', (~valid_salary).sum())
print('Duplicate-like rows:', df.duplicated(subset=['name', 'position', 'start_date', 'salary']).sum())

df.loc[~valid_name, 'name'] = pd.NA
df.loc[~valid_position, 'position'] = pd.NA
df.loc[~valid_salary, 'salary'] = pd.NA
df = df.dropna(subset=['employee_id', 'name', 'position', 'start_date', 'salary'])
df = df.drop_duplicates(subset=['name', 'position', 'start_date', 'salary'], keep='first').copy()

df.head()

Invalid names: 6
Invalid positions: 3
Invalid dates: 0
Invalid salaries: 0
Duplicate-like rows: 1


,employee_id,name,position,start_date,salary
0,6,Amy Jordan,Logistics Coordinator,2024-05-10,71529.0
1,7,Angela Haynes,Sales Representative,2018-11-03,84555.0
2,8,Adam Grant,Hr Specialist,2018-12-03,111760.0
3,10,Cameron Mcgee,Marketing Manager,2024-12-04,92045.0
4,11,Donna Rivera,Logistics Coordinator,2024-01-25,141098.0


## Data Transformation

The `start_date` column is standardized as a datetime. The position text is trimmed and normalized to title case so equivalent labels have a consistent format. A `start_year` column is extracted to support time-based summaries.

In [ ]:
df['position'] = df['position'].str.strip().str.title()
df['start_year'] = df['start_date'].dt.year
df[['employee_id', 'position', 'start_date', 'start_year', 'salary']].head()

,employee_id,position,start_date,start_year,salary
0,6,Logistics Coordinator,2024-05-10,2024,71529.0
1,7,Sales Representative,2018-11-03,2018,84555.0
2,8,Hr Specialist,2018-12-03,2018,111760.0
3,10,Marketing Manager,2024-12-04,2024,92045.0
4,11,Logistics Coordinator,2024-01-25,2024,141098.0


## Feature Engineering

A `years_of_service` feature is derived from each employee's start date. It uses the current date and is clipped at zero so future start dates do not produce negative service values.

In [ ]:
today = pd.Timestamp.today().normalize()
df['years_of_service'] = ((today - df['start_date']).dt.days / 365.25).clip(lower=0).round(1)
df[['name', 'position', 'start_date', 'years_of_service']].head()

,name,position,start_date,years_of_service
0,Daryl Carrillo,Graphic Designer,2022-02-12,4.6
1,Shawn Frost,Accountant,2024-09-10,2.0
2,Justin Lewis,Sales Representative,2021-03-25,5.5
3,Joseph Lopez,Sales Representative,2021-07-07,5.2
4,Andre Long,Customer Support Representative,2015-01-14,11.7


## Scaling

Salary is min-max scaled to the range 0 to 1. This preserves the relative ordering of salaries while putting the numeric feature on a comparable scale for analysis or machine-learning models. The original `salary` column is retained for interpretation.

In [ ]:
salary_min = df['salary'].min()
salary_range = df['salary'].max() - salary_min
df['salary_scaled'] = (df['salary'] - salary_min) / salary_range if salary_range else 0.0
df[['salary', 'salary_scaled', 'years_of_service']].head()

,salary,salary_scaled,years_of_service
0,113317,0.383278,4.6
1,60975,0.000000,2.0
2,131670,0.517669,5.5
3,115438,0.398809,5.2
4,121870,0.445908,11.7
